<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day1_3_(260323)_js_%EC%9D%B4%EB%B2%A4%ED%8A%B8_%EC%8B%9C%EC%8A%A4%ED%85%9C%EA%B3%BC_%EB%B9%84%EB%8F%99%EA%B8%B0_%EC%A0%9C%EC%96%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JavaScript 이벤트 시스템과 비동기 제어

In [2]:
%%writefile /content/server.js

const express = require('express');
const path = require('path');

const app = express();
const PORT = 3000;

// JSON 데이터 처리를 위한 미들웨어
app.use(express.json());

// 1. [이벤트 시스템의 시작] 메인 페이지 제공
app.get('/', (req, res) => {
    // templates 폴더 안의 index.html을 읽어서 브라우저로 전송
    res.sendFile(path.join(__dirname, 'templates', 'index.html'));
});

// 2. [실행 타이밍 시스템의 대상] 비동기 요청 API
app.get('/api/data', async (req, res) => {
    try {
        // 실제 DB 조회를 시뮬레이션하기 위해 1초의 지연 시간을 둠
        await new Promise(resolve => setTimeout(resolve, 1000));

    const result = {
        status: "success",
        message: "서버 데이터 수신 완료",
        updateLog: "v1.2.5 패치 적용됨", // 추가
        serverTime: new Date().toLocaleString('ko-KR'),
        dataList: ['이렇게 하면 데이터가 잘 전달 되나?','전달되겠지','그렇지?']
    };

        res.json(result);
    } catch (error) {
        res.status(500).json({ message: "서버 내부 실행 오류" });
    }
});

app.listen(PORT, () => { console.log(`Server runing on:${PORT}`); });

Overwriting /content/server.js


In [3]:
%%writefile /content/templates/index.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>이벤트 & 비동기 시스템</title>
    <style>
        body { font-family: 'Pretendard', sans-serif; background: #f0f2f5; display: flex; justify-content: center; align-items: center; height: 100vh; margin: 0; }
        .card { background: white; padding: 2rem; border-radius: 16px; box-shadow: 0 10px 25px rgba(0,0,0,0.1); width: 100%; max-width: 500px; }
        h1 { font-size: 1.5rem; color: #1c1e21; margin-bottom: 1.5rem; text-align: center; }
        button { width: 100%; padding: 12px; background: #0064ff; color: white; border: none; border-radius: 8px; font-size: 1rem; font-weight: bold; cursor: pointer; transition: background 0.2s; }
        button:hover { background: #0052d6; }
        button:disabled { background: #ccc; cursor: not-allowed; }
        #display { margin-top: 1.5rem; padding: 1rem; background: #f8f9fa; border: 1px solid #e9ecef; border-radius: 8px; min-height: 120px; font-size: 0.9rem; line-height: 1.6; color: #4b4f56; white-space: pre-line; }
        .loading-text { color: #0064ff; font-weight: bold; }
    </style>
</head>
<body>

<div class="card">
    <h1>시스템 실행 실습</h1>
    <button id="fetchBtn">데이터 요청 실행 (Click)</button>

    <div id="display">버튼을 클릭하여 이벤트를 발생시키세요.</div>
</div>

<script>
    // 1. 객체 선택 (DOM 탐색)
    const fetchBtn = document.getElementById('fetchBtn');
    const display = document.getElementById('display');

    /**
     * [핵심 1: 이벤트 시스템]
     * 객체(fetchBtn)에 조건(click)과 핸들러(async function)를 등록함
     */
    fetchBtn.addEventListener('click', async () => {
        try {
            // UI 상태 변경 (사용자에게 피드백)
            fetchBtn.disabled = true;
            display.innerHTML = '<span class="loading-text"> 실행 타이밍 시스템: 대기열(Queue) 진입 후 처리 중...</span>';

            /**
             * [핵심 2: async/await 비동기 제어]
             * await를 만나는 순간 이 함수의 실행은 '일시 중단'됨.
             * 브라우저는 이 동안 멈추지 않고 다른 작업을 할 수 있음.
             */
            const response = await fetch('/api/data');

            if (!response.ok) throw new Error('서버 응답 실패');

            // 데이터를 JS 객체로 변환 (이 또한 비동기 작업)
            const result = await response.json();

            /**
             * [핵심 3: 실행 재개]
             * 응답이 도착하면 중단되었던 지점부터 다시 실행됨.
             */
            display.textContent = `수신 성공!\n\n` +
                                 `메시지: ${result.message}\n` +
                                 `버전 정보: ${result.updateLog}\n`+
                                 `데이터: ${result.dataList.join(', ')}`;

        } catch (error) {
            display.textContent = `오류 발생: ${error.message}`;
        } finally {
            fetchBtn.disabled = false;
        }
    });
</script>

</body>
</html>

Overwriting /content/templates/index.html


# 충정일보

In [1]:
%%writefile server.js


const express = require('express');
const path = require('path');
const fs = require('fs');
const mysql = require('mysql2/promise');
const jwt = require('jsonwebtoken');
const bcrypt = require('bcrypt');
const multer = require('multer');

const app = express();
const PORT = 3000;
const JWT_SECRET = 'my_super_secret_key';

app.use(express.json());
app.use(express.urlencoded({ extended: true }));

const uploadDir = path.join(__dirname, 'uploads');
const staticDir = path.join(__dirname, 'static');
const cssDir = path.join(staticDir, 'css');
const imagesDir = path.join(staticDir, 'images');

if (!fs.existsSync(uploadDir)) fs.mkdirSync(uploadDir);
if (!fs.existsSync(staticDir)) fs.mkdirSync(staticDir);
if (!fs.existsSync(cssDir)) fs.mkdirSync(cssDir);
if (!fs.existsSync(imagesDir)) fs.mkdirSync(imagesDir);

app.use('/uploads', express.static(uploadDir));
app.use('/static', express.static(staticDir));

const storage = multer.diskStorage({
    destination: (req, file, cb) => cb(null, uploadDir),
    filename: (req, file, cb) => {
        const ext = path.extname(file.originalname);
        const fileName = Date.now() + '_' + Math.round(Math.random() * 1e9) + ext;
        cb(null, fileName);
    }
});
const upload = multer({ storage });

const pool = mysql.createPool({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb'
});

function getUserFromToken(req) {
    const authHeader = req.headers['authorization'];
    const token = authHeader && authHeader.split(' ')[1];
    if (!token) return null;

    try {
        return jwt.verify(token, JWT_SECRET);
    } catch (err) {
        return null;
    }
}

// 페이지 라우팅
app.get('/', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'login.html')));
app.get('/signup', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'signup.html')));
app.get('/main', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'main.html')));
app.get('/subscriber', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'subscriber.html')));
app.get('/reporter', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'reporter.html')));
app.get('/admin', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'admin.html')));
app.get('/article', (req, res) => res.sendFile(path.join(__dirname, 'templates', 'article.html')));

// 회원가입
app.post('/api/signup', async (req, res) => {
    try {
        const { userId, password, userName, role } = req.body;

        if (!userId || !password || !userName || !role) {
            return res.status(400).json({ success: false, message: '입력값 누락' });
        }

        if (!['subscriber', 'reporter', 'admin'].includes(role)) {
            return res.status(400).json({ success: false, message: '역할 오류' });
        }

        const hashed = await bcrypt.hash(password, 10);

        await pool.execute(
            'INSERT INTO users (user_id, password, user_name, role) VALUES (?, ?, ?, ?)',
            [userId, hashed, userName, role]
        );

        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false, message: '회원가입 실패' });
    }
});

// 로그인
app.post('/api/login', async (req, res) => {
    try {
        const { userId, password } = req.body;
        const [rows] = await pool.execute('SELECT * FROM users WHERE user_id = ?', [userId]);
        const user = rows[0];

        if (user && await bcrypt.compare(password, user.password)) {
            const token = jwt.sign(
                {
                    id: user.id,
                    userId: user.user_id,
                    userName: user.user_name,
                    role: user.role
                },
                JWT_SECRET,
                { expiresIn: '1h' }
            );

            res.json({ success: true, token });
        } else {
            res.status(401).json({ success: false, message: '로그인 실패' });
        }
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false, message: '서버 오류' });
    }
});

// JWT 검증
app.get('/api/verify', (req, res) => {
    const authHeader = req.headers['authorization'];
    const token = authHeader && authHeader.split(' ')[1];

    if (!token) return res.json({ success: false });

    jwt.verify(token, JWT_SECRET, (err, decoded) => {
        if (err) return res.json({ success: false });
        res.json({ success: true, user: decoded });
    });
});

// 구독자용 기사 조회: published만 노출, 검색 포함
app.get('/api/data', async (req, res) => {
    try {
        const search = req.query.search ? req.query.search.trim() : '';
        let sql = `
            SELECT id, title, content, article_date, category, reporter_name, photo_path, status, created_at
            FROM articles
            WHERE status = 'published'
        `;
        const params = [];

        if (search) {
            sql += ` AND (title LIKE ? OR content LIKE ? OR reporter_name LIKE ? OR category LIKE ?)`;
            const keyword = `%${search}%`;
            params.push(keyword, keyword, keyword, keyword);
        }

        sql += ` ORDER BY id DESC`;

        const [rows] = await pool.execute(sql, params);
        res.json(rows);
    } catch (err) {
        console.error(err);
        res.status(500).json([]);
    }
});

// 기사 상세
app.get('/api/article/:id', async (req, res) => {
    try {
        const user = getUserFromToken(req);

        const [rows] = await pool.execute(`
            SELECT id, title, content, article_date, category, reporter_id, reporter_name, photo_path, status, created_at
            FROM articles
            WHERE id = ?
        `, [req.params.id]);

        if (rows.length === 0) {
            return res.status(404).json({ success: false });
        }

        const article = rows[0];

        if (article.status !== 'published') {
            if (!user) {
                return res.status(403).json({ success: false, message: '미출판 기사입니다.' });
            }

            const canRead =
                user.role === 'admin' ||
                (user.role === 'reporter' && user.id === article.reporter_id);

            if (!canRead) {
                return res.status(403).json({ success: false, message: '미출판 기사입니다.' });
            }
        }

        res.json({ success: true, article });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false });
    }
});

// 기자 기사 등록
app.post('/api/reporter/article', upload.single('photo'), async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json({ success: false, message: '인증 실패' });
        if (user.role !== 'reporter') return res.status(403).json({ success: false, message: '기자 전용 기능' });

        const { title, content, articleDate, category } = req.body;
        const photoPath = req.file ? '/uploads/' + req.file.filename : '/static/images/default-photo.svg';

        await pool.execute(`
            INSERT INTO articles (title, content, article_date, category, reporter_id, reporter_name, photo_path, status)
            VALUES (?, ?, ?, ?, ?, ?, ?, 'pending')
        `, [title, content, articleDate, category, user.id, user.userName, photoPath]);

        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false, message: '기사 등록 실패' });
    }
});

// 기자 본인 기사 조회 + 검색
app.get('/api/reporter/my-articles', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json([]);
        if (user.role !== 'reporter') return res.status(403).json([]);

        const search = req.query.search ? req.query.search.trim() : '';
        let sql = `
            SELECT id, title, content, article_date, category, reporter_name, photo_path, status, created_at
            FROM articles
            WHERE reporter_id = ?
        `;
        const params = [user.id];

        if (search) {
            sql += ` AND (title LIKE ? OR content LIKE ? OR category LIKE ?)`;
            const keyword = `%${search}%`;
            params.push(keyword, keyword, keyword);
        }

        sql += ` ORDER BY id DESC`;

        const [rows] = await pool.execute(sql, params);
        res.json(rows);
    } catch (err) {
        console.error(err);
        res.status(500).json([]);
    }
});

// 기자 기사 수정
app.put('/api/reporter/article/:id', upload.single('photo'), async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json({ success: false });
        if (user.role !== 'reporter') return res.status(403).json({ success: false });

        const articleId = req.params.id;
        const { title, content, articleDate, category } = req.body;

        const [rows] = await pool.execute(
            'SELECT * FROM articles WHERE id = ? AND reporter_id = ?',
            [articleId, user.id]
        );

        if (rows.length === 0) {
            return res.status(404).json({ success: false, message: '기사 없음' });
        }

        const article = rows[0];
        const photoPath = req.file ? '/uploads/' + req.file.filename : article.photo_path;

        await pool.execute(`
            UPDATE articles
            SET title = ?, content = ?, article_date = ?, category = ?, photo_path = ?, status = 'pending'
            WHERE id = ? AND reporter_id = ?
        `, [title, content, articleDate, category, photoPath, articleId, user.id]);

        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false });
    }
});

// 기자 기사 삭제
app.delete('/api/reporter/article/:id', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json({ success: false });
        if (user.role !== 'reporter') return res.status(403).json({ success: false });

        await pool.execute(
            'DELETE FROM articles WHERE id = ? AND reporter_id = ?',
            [req.params.id, user.id]
        );

        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false });
    }
});

// 관리자 사용자 조회
app.get('/api/admin/users', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json([]);
        if (user.role !== 'admin') return res.status(403).json([]);

        const [rows] = await pool.execute(`
            SELECT id, user_id, user_name, role
            FROM users
            ORDER BY id DESC
        `);

        res.json(rows);
    } catch (err) {
        console.error(err);
        res.status(500).json([]);
    }
});

// 관리자 사용자 삭제
app.delete('/api/admin/users/:id', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json({ success: false });
        if (user.role !== 'admin') return res.status(403).json({ success: false });

        await pool.execute('DELETE FROM users WHERE id = ?', [req.params.id]);
        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false });
    }
});

// 관리자 아이디 수정
app.put('/api/admin/users/:id/userid', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json({ success: false });
        if (user.role !== 'admin') return res.status(403).json({ success: false });

        const { userId } = req.body;
        await pool.execute('UPDATE users SET user_id = ? WHERE id = ?', [userId, req.params.id]);

        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false });
    }
});

// 관리자 비밀번호 수정
app.put('/api/admin/users/:id/password', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json({ success: false });
        if (user.role !== 'admin') return res.status(403).json({ success: false });

        const { password } = req.body;
        const hashed = await bcrypt.hash(password, 10);

        await pool.execute('UPDATE users SET password = ? WHERE id = ?', [hashed, req.params.id]);
        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false });
    }
});

// 관리자 기사 목록 조회 + 검색
app.get('/api/admin/articles', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json([]);
        if (user.role !== 'admin') return res.status(403).json([]);

        const search = req.query.search ? req.query.search.trim() : '';
        let sql = `
            SELECT id, title, content, article_date, category, reporter_name, photo_path, status, created_at
            FROM articles
            WHERE 1=1
        `;
        const params = [];

        if (search) {
            sql += ` AND (title LIKE ? OR content LIKE ? OR reporter_name LIKE ? OR category LIKE ? OR status LIKE ?)`;
            const keyword = `%${search}%`;
            params.push(keyword, keyword, keyword, keyword, keyword);
        }

        sql += ` ORDER BY id DESC`;

        const [rows] = await pool.execute(sql, params);
        res.json(rows);
    } catch (err) {
        console.error(err);
        res.status(500).json([]);
    }
});

// 관리자 승인
app.put('/api/admin/articles/:id/publish', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json({ success: false });
        if (user.role !== 'admin') return res.status(403).json({ success: false });

        await pool.execute(
            "UPDATE articles SET status = 'published' WHERE id = ?",
            [req.params.id]
        );

        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false });
    }
});

// 관리자 반려
app.put('/api/admin/articles/:id/reject', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json({ success: false });
        if (user.role !== 'admin') return res.status(403).json({ success: false });

        await pool.execute(
            "UPDATE articles SET status = 'rejected' WHERE id = ?",
            [req.params.id]
        );

        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false });
    }
});

// 댓글 조회
app.get('/api/article/:id/comments', async (req, res) => {
    try {
        const [rows] = await pool.execute(`
            SELECT id, user_id, user_name, content, created_at
            FROM comments
            WHERE article_id = ?
            ORDER BY id DESC
        `, [req.params.id]);

        res.json(rows);
    } catch (err) {
        console.error(err);
        res.status(500).json([]);
    }
});

// 댓글 등록
app.post('/api/article/:id/comments', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json({ success: false, message: '로그인 필요' });

        const { content } = req.body;
        if (!content || !content.trim()) {
            return res.status(400).json({ success: false, message: '댓글 내용 없음' });
        }

        const [rows] = await pool.execute(`
            SELECT id, status
            FROM articles
            WHERE id = ?
        `, [req.params.id]);

        if (rows.length === 0) {
            return res.status(404).json({ success: false });
        }

        if (rows[0].status !== 'published') {
            return res.status(403).json({ success: false, message: '출판 기사만 댓글 가능' });
        }

        await pool.execute(`
            INSERT INTO comments (article_id, user_id, user_name, content)
            VALUES (?, ?, ?, ?)
        `, [req.params.id, user.id, user.userName, content.trim()]);

        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false });
    }
});

// 댓글 삭제: 관리자 또는 작성자
app.delete('/api/comments/:id', async (req, res) => {
    try {
        const user = getUserFromToken(req);
        if (!user) return res.status(401).json({ success: false });

        const [rows] = await pool.execute(
            'SELECT * FROM comments WHERE id = ?',
            [req.params.id]
        );

        if (rows.length === 0) {
            return res.status(404).json({ success: false });
        }

        const comment = rows[0];
        const canDelete = user.role === 'admin' || user.id === comment.user_id;

        if (!canDelete) {
            return res.status(403).json({ success: false });
        }

        await pool.execute('DELETE FROM comments WHERE id = ?', [req.params.id]);
        res.json({ success: true });
    } catch (err) {
        console.error(err);
        res.status(500).json({ success: false });
    }
});

app.listen(PORT, () => console.log(`서버 실행 중: http://localhost:${PORT}`));


Writing server.js


# 공통

In [2]:
!npm init -y

Wrote to /content/package.json:

{
  "name": "content",
  "version": "1.0.0",
  "main": "server.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1",
    "start": "node server.js"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "description": ""
}



⠙

In [3]:
!npm install express mysql2

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
added 76 packages, and audited 77 packages in 5s
⠴
⠴24 packages are looking for funding
⠴  run `npm fund` for details
⠴
found 0 vulnerabilities
⠴

In [4]:
!npm install express mysql2 jsonwebtoken bcrypt multer

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 31 packages, and audited 108 packages in 3s
⠹
⠹26 packages are looking for funding
⠹  run `npm fund` for details
⠹
found 0 vulnerabilities
⠹

In [5]:
!nohup node server.js > server.log 2>&1 &

In [6]:
!tail -n 20 server.log

In [13]:
!npm install -g cloudflared

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
changed 1 package in 1s
⠹

In [14]:
!nohup cloudflared tunnel --url http://localhost:3000 > tunnel.log 2>&1 &

In [15]:
!tail -n 20 tunnel.log

2026-03-23T05:50:17Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-23T05:50:17Z INF Requesting new quick Tunnel on trycloudflare.com...


In [16]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" tunnel.log |tail -n 1

https://economies-adopted-del-detailed.trycloudflare.com
